In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. LOAD THE SMS SPAM DATASET
# – Download “SMSSpamCollection” from UCI (https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection)

df = pd.read_csv('/home/SMSSpamCollection', sep='\t', names=['label','message'])
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
# 2. PREPROCESS & VECTORIZE TEXT

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.95,
    min_df=5
)
X = vectorizer.fit_transform(df['message'])

y = df['label']

# 3. TRAIN/TEST SPLIT (stratify to keep spam/ham ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# 4. HYPERPARAMETER TUNING WITH GRIDSEARCHCV
param_grid = {
    'criterion':        ['gini', 'entropy'],
    'max_depth':        [None, 10, 20, 30],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}


grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',      # since dataset is imbalanced, F1-score is more informative
    n_jobs=-1          #This tells GridSearchCV to use all CPU cores for faster computation.
)


grid.fit(X_train, y_train)

best_dt = grid.best_estimator_
print("Best hyperparameters:", grid.best_params_)
print("Best CV F1-score :", grid.best_score_)

# 5. FINAL EVALUATION ON HOLD-OUT TEST SET
y_pred = best_dt.predict(X_test)
print("\nTest Accuracy       :", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n",
      classification_report(y_test, y_pred, target_names=['ham','spam']))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 6. FEATURE IMPORTANCES (Top 10 terms driving “spam” decisions)
#To show the Top 10 most important words (features) that the Decision Tree model is using to classify emails as spam or not.


importances = best_dt.feature_importances_

features   = np.array(vectorizer.get_feature_names_out())


top_idx    = np.argsort(importances)[-10:][::-1]


print("\nTop 10 Important Terms:")
for idx in top_idx:
    print(f"  {features[idx]:<15}  →  {importances[idx]:.4f}")



Best hyperparameters: {'criterion': 'entropy', 'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV F1-score : 0.8514145215354636

Test Accuracy       : 0.9605263157894737

Classification Report:
               precision    recall  f1-score   support

         ham       0.97      0.99      0.98      1448
        spam       0.90      0.79      0.84       224

    accuracy                           0.96      1672
   macro avg       0.94      0.89      0.91      1672
weighted avg       0.96      0.96      0.96      1672

Confusion Matrix:
 [[1429   19]
 [  47  177]]

Top 10 Important Terms:
  txt              →  0.1520
  claim            →  0.1047
  free             →  0.0952
  www              →  0.0681
  service          →  0.0544
  landline         →  0.0371
  reply            →  0.0341
  150p             →  0.0340
  50               →  0.0334
  min              →  0.0319
